<a href="https://colab.research.google.com/github/irullah/comic-yolo/blob/main/Training_AI_Comic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install ultralytics roboflow

In [ ]:
import os
import cv2
import glob
import yaml
from roboflow import Roboflow
from ultralytics import YOLO

# ==============================================================================
# TAHAP 1 & 2: UNDUH DATASET
# ==============================================================================
print("[INFO] Mengunduh dataset dari Roboflow...")
rf = Roboflow(api_key="w7A0SzgQrYyWSGqS4CaX")
project = rf.workspace("poos-workspace-20cho").project("ruru-u8tn6")
version = project.version(1)
dataset = version.download("yolov8")

# ==============================================================================
# TAHAP 3: PEMOTONGAN GAMBAR PANJANG MANUAL (BEBAS BUG)
# ==============================================================================
print("\n[INFO] Memotong gambar komik panjang menjadi kotak-kotak (Manual Script)...")

src_dir = dataset.location
dst_dir = f"{dataset.location}_tiled"
os.makedirs(dst_dir, exist_ok=True)

# 1. Buat folder untuk dataset baru
for split in ['train', 'valid', 'test']:
    os.makedirs(os.path.join(dst_dir, split, 'images'), exist_ok=True)
    os.makedirs(os.path.join(dst_dir, split, 'labels'), exist_ok=True)

# 2. Proses potong gambar dan poligon secara matematis
for split in ['train', 'valid', 'test']:
    img_paths = glob.glob(os.path.join(src_dir, split, 'images', '*.*'))
    for img_path in img_paths:
        img = cv2.imread(img_path)
        if img is None: continue

        H, W = img.shape[:2]
        slice_h = W  # Potong menjadi bujur sangkar mengikuti lebar komik

        # Baca label poligon dari Roboflow
        label_path = img_path.replace('images', 'labels').rsplit('.', 1)[0] + '.txt'
        polygons = []
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f.readlines():
                    parts = line.strip().split()
                    if len(parts) < 3: continue
                    cls_id = parts[0]
                    coords = [float(p) for p in parts[1:]]
                    # Ubah koordinat relatif (0-1) ke absolut (pixel)
                    abs_coords = [(coords[i] * W, coords[i+1] * H) for i in range(0, len(coords), 2)]
                    polygons.append({'cls': cls_id, 'pts': abs_coords})

        y_start = 0
        slice_idx = 0
        while y_start < H:
            y_end = min(y_start + slice_h, H)
            actual_h = y_end - y_start
            if actual_h < slice_h * 0.1: break # Abaikan sisa potongan mikroskopis di paling bawah

            crop = img[y_start:y_end, 0:W]
            crop_polys = []

            for poly in polygons:
                new_pts = []
                is_inside = False
                for (x, y) in poly['pts']:
                    if y_start <= y <= y_end:
                        is_inside = True
                    # Jepit koordinat poligon agar tidak keluar dari batas gambar terpotong
                    nx = max(0, min(x, W))
                    ny = max(0, min(y - y_start, actual_h))
                    new_pts.append((nx, ny))

                if is_inside:
                    # Kembalikan ke format koordinat persentase (0-1) standar YOLO
                    norm_pts = [f"{nx/W:.6f} {ny/actual_h:.6f}" for (nx, ny) in new_pts]
                    crop_polys.append(f"{poly['cls']} " + " ".join(norm_pts))

            # Simpan gambar
            base_name = os.path.basename(img_path).rsplit('.', 1)[0]
            cv2.imwrite(os.path.join(dst_dir, split, 'images', f"{base_name}_{slice_idx}.jpg"), crop)

            # Simpan label teksnya
            if crop_polys:
                with open(os.path.join(dst_dir, split, 'labels', f"{base_name}_{slice_idx}.txt"), 'w') as f:
                    f.write("\n".join(crop_polys) + "\n")

            y_start += slice_h - int(slice_h * 0.1)  # Beri overlap 10% agar teks tidak terbelah di tengah jalan
            slice_idx += 1

# 3. Buat file data.yaml baru agar AI tidak tersesat
if os.path.exists(os.path.join(src_dir, 'data.yaml')):
    with open(os.path.join(src_dir, 'data.yaml'), 'r') as f:
        data_yaml = yaml.safe_load(f)

    data_yaml['path'] = os.path.abspath(dst_dir) # Kunci utama agar YOLO mengenali lokasi baru
    data_yaml['train'] = 'train/images'
    data_yaml['val'] = 'valid/images'
    if 'test' in data_yaml: data_yaml['test'] = 'test/images'

    with open(os.path.join(dst_dir, 'data.yaml'), 'w') as f:
        yaml.dump(data_yaml, f)

print("[INFO] Pemotongan selesai dan poligon berhasil diamankan dengan sempurna! Lanjut ke Cell 2.\n")

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
[INFO] Mengunduh dataset dari Roboflow...
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Ruru-1 in yolov8:: 100%|██████████| 609/609 [00:00<00:00, 7720.32it/s]



[INFO] Memotong gambar komik panjang menjadi kotak-kotak (Manual Script)...
[INFO] Pemotongan selesai dan poligon berhasil diamankan dengan sempurna! Lanjut ke Cell 2.



In [ ]:
import os
import yaml
import glob
from ultralytics import YOLO

# 1. Cari folder dataset yang sudah dipotong tadi (Ruru-1_tiled)
tiled_dirs = glob.glob("/content/*_tiled")

if tiled_dirs:
    dst_dir = tiled_dirs[0]
    yaml_path = os.path.join(dst_dir, "data.yaml")

    # 2. Buka dan perbaiki konfigurasi data.yaml
    with open(yaml_path, 'r') as f:
        data_yaml = yaml.safe_load(f)

    val_images = glob.glob(os.path.join(dst_dir, 'valid', 'images', '*.*'))

    # Jika folder validasi terbukti kosong, alihkan ke folder train
    if len(val_images) == 0:
        print("🔧 [PERBAIKAN] Folder 'valid' kosong. AI akan dites menggunakan data 'train'...")
        data_yaml['val'] = 'train/images'

        with open(yaml_path, 'w') as f:
            yaml.dump(data_yaml, f)
    else:
        print("✅ [INFO] Data validasi ditemukan!")

    # ==============================================================================
    # 3. LANJUTKAN TRAINING (TANPA MENGULANG DARI AWAL)
    # ==============================================================================
    print("🚀 [INFO] Menyalakan ulang mesin Training YOLOv8...\n")
    model = YOLO("yolov8n-seg.pt")

    results = model.train(
        data=yaml_path,
        epochs=50,
        imgsz=640,
        batch=16,
        project="Comic_Cleaner",
        name="Teks_Segmenter"
    )
    print("\n🎉 === TRAINING SELESAI! ===")

else:
    print("❌ [ERROR] Folder dataset tidak ditemukan. Pastikan Anda tidak menghapus folder /content/")

🔧 [PERBAIKAN] Folder 'valid' kosong. AI akan dites menggunakan data 'train'...
🚀 [INFO] Menyalakan ulang mesin Training YOLOv8...

Ultralytics 8.4.115 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Ruru-1_tiled/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max

In [ ]:
import os
import zipfile
from google.colab import files # Wajib ditambahkan untuk fitur download

# Tentukan nama file hasil kompresi
zip_filename = 'semua_data_ruru.zip'

# Daftar file dan folder yang ingin di-zip
items_to_zip = ['runs', 'Ruru-1', 'Ruru-1_tiled', 'yolo26n.pt', 'yolov8n-seg.pt']

print("Sedang mengompres semua file dan folder...")

# Proses kompresi
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for item in items_to_zip:
        if os.path.isdir(item):
            # Jika item berupa folder, masukkan semua isinya
            for root, dirs, filenames in os.walk(item):
                for filename in filenames:
                    file_path = os.path.join(root, filename)
                    zipf.write(file_path, file_path)
        elif os.path.exists(item):
            # Jika item berupa file tunggal
            zipf.write(item, item)

print(f"Proses kompresi selesai! File {zip_filename} sedang diunduh...")

# Memulai proses download otomatis ke komputermu
files.download(zip_filename)

Sedang mengompres semua file dan folder...
Proses kompresi selesai! File semua_data_ruru.zip sedang diunduh...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>